# 06 — Xception + Classical ML Ensemble

Uses the Xception model from `05_Xception.ipynb` as a feature extractor. Input size: 299×299.

## Section 1: Imports & Configuration

In [ ]:
import os
import random
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras.models import load_model, Model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix,
                             roc_curve, auc, precision_recall_curve,
                             average_precision_score)
from sklearn.preprocessing import label_binarize
import joblib
print("✓ Imports complete")

## Section 2: Constants & Hyperparameters

In [ ]:
NOTEBOOK_NAME    = "06_Xception_Ensemble"

DATASET_PATH     = "../MRI_DATASET/"
TRAIN_DIR        = DATASET_PATH + "Training/"
TEST_DIR         = DATASET_PATH + "Testing/"

CLASS_NAMES      = ['glioma', 'meningioma', 'notumor', 'pituitary']
NUM_CLASSES      = 4

IMG_HEIGHT       = 299  # InceptionV3/Xception requires 299×299
IMG_WIDTH        = 299
CHANNELS         = 3

BATCH_SIZE       = 32
LEARNING_RATE    = 1e-4
VALIDATION_SPLIT = 0.2
RANDOM_SEED      = 42

SAVED_MODELS_DIR = "../saved_models/"
os.makedirs(SAVED_MODELS_DIR, exist_ok=True)

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)
os.environ['PYTHONHASHSEED'] = str(RANDOM_SEED)

print("✓ Constants configured")
print(f"  Image size : {IMG_HEIGHT}×{IMG_WIDTH}")
print(f"  Classes    : {CLASS_NAMES}")

## Section 3: Data Loading & Verification

In [ ]:
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.1,
    zoom_range=0.1,
    horizontal_flip=True,
    fill_mode='nearest',
    validation_split=VALIDATION_SPLIT
)
test_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    classes=CLASS_NAMES,
    subset='training',
    seed=RANDOM_SEED,
    shuffle=True
)
val_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    classes=CLASS_NAMES,
    subset='validation',
    seed=RANDOM_SEED,
    shuffle=False
)
test_generator = test_datagen.flow_from_directory(
    TEST_DIR,
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    classes=CLASS_NAMES,
    shuffle=False
)

# No-augmentation generator for feature extraction (deterministic output)
feature_datagen = ImageDataGenerator(rescale=1./255)
feature_train_generator = feature_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    classes=CLASS_NAMES,
    shuffle=False
)

print("=" * 50)
print("DATA VERIFICATION")
print(f"Class indices      : {train_generator.class_indices}")
print(f"Training samples   : {train_generator.samples}")
print(f"Validation samples : {val_generator.samples}")
print(f"Test samples       : {test_generator.samples}")
print(f"Image size         : {IMG_HEIGHT}×{IMG_WIDTH}")
print(f"Batch size         : {BATCH_SIZE}")
print("=" * 50)

## Section 4: Data Preprocessing & Augmentation

In [ ]:
# Preprocessing: rescale 1/255 via ImageDataGenerator.
print("✓ Preprocessing configured via ImageDataGenerator (rescale 1/255)")

## Section 5: Model Definition

In [ ]:
xcp_model_path = SAVED_MODELS_DIR + 'xception_model.h5'
xcp_base = load_model(xcp_model_path)
print(f"✓ Xception model loaded from: {xcp_model_path}")

feature_extractor = Model(
    inputs=xcp_base.input,
    outputs=xcp_base.get_layer('feature_layer').output,
    name='xception_feature_extractor'
)
print(f"✓ Feature extractor — output shape: {feature_extractor.output_shape}")

rf  = RandomForestClassifier(n_estimators=100, random_state=RANDOM_SEED, n_jobs=-1)
dt  = DecisionTreeClassifier(random_state=RANDOM_SEED)
svm = SVC(probability=True, random_state=RANDOM_SEED, kernel='rbf', C=1.0)
ensemble_clf = VotingClassifier(
    estimators=[('rf', rf), ('dt', dt), ('svm', svm)],
    voting='soft'
)
print("✓ Classifiers defined: RF, DT, SVM, Soft-Vote Ensemble")

## Section 6: Model Training

In [ ]:
print("Extracting training features...")
feature_train_generator.reset()
X_train = feature_extractor.predict(feature_train_generator, verbose=1)
y_train = feature_train_generator.classes
print(f"✓ Training features: {X_train.shape}")

print("Extracting test features...")
test_generator.reset()
X_test = feature_extractor.predict(test_generator, verbose=1)
y_test  = test_generator.classes
print(f"✓ Test features: {X_test.shape}")

print("\nTraining Random Forest...")
rf.fit(X_train, y_train)
print("✓ Random Forest trained")

print("Training Decision Tree...")
dt.fit(X_train, y_train)
print("✓ Decision Tree trained")

print("Training SVM...")
svm.fit(X_train, y_train)
print("✓ SVM trained")

print("Training Soft-Vote Ensemble...")
ensemble_clf.fit(X_train, y_train)
print("✓ Ensemble trained")

## Section 7: Model Evaluation

In [ ]:
def evaluate_sklearn_model(clf, X_test, y_test, model_name="Model"):
    """Standard evaluation for sklearn classifiers."""    y_pred       = clf.predict(X_test)
    y_pred_proba = clf.predict_proba(X_test)
    acc = accuracy_score(y_test, y_pred)

    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
    plt.title(f'{model_name} — Confusion Matrix (Acc: {acc:.4f})')
    plt.ylabel('Actual')
    plt.xlabel('Predicted')
    plt.tight_layout()
    plt.show()

    print(f"\n{model_name} — Classification Report")
    print("=" * 60)
    print(classification_report(y_test, y_pred, target_names=CLASS_NAMES))

    y_true_bin = label_binarize(y_test, classes=[0, 1, 2, 3])
    plt.figure(figsize=(8, 6))
    for i, cls in enumerate(CLASS_NAMES):
        fpr, tpr, _ = roc_curve(y_true_bin[:, i], y_pred_proba[:, i])
        roc_auc = auc(fpr, tpr)
        plt.plot(fpr, tpr, label=f'{cls} (AUC = {roc_auc:.2f})')
    plt.plot([0, 1], [0, 1], 'k--')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title(f'{model_name} — ROC Curve')
    plt.legend(loc='lower right')
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(8, 6))
    for i, cls in enumerate(CLASS_NAMES):
        precision, recall, _ = precision_recall_curve(y_true_bin[:, i], y_pred_proba[:, i])
        ap = average_precision_score(y_true_bin[:, i], y_pred_proba[:, i])
        plt.plot(recall, precision, label=f'{cls} (AP = {ap:.2f})')
    plt.xlabel('Recall')
    plt.ylabel('Precision')
    plt.title(f'{model_name} — Precision-Recall Curve')
    plt.legend(loc='upper right')
    plt.tight_layout()
    plt.show()

    return acc, y_pred, y_pred_proba

rf_acc,  rf_pred,  rf_proba  = evaluate_sklearn_model(rf,  X_test, y_test, "Xception + Random Forest")
dt_acc,  dt_pred,  dt_proba  = evaluate_sklearn_model(dt,  X_test, y_test, "Xception + Decision Tree")
svm_acc, svm_pred, svm_proba = evaluate_sklearn_model(svm, X_test, y_test, "Xception + SVM")
ens_acc, ens_pred, ens_proba = evaluate_sklearn_model(ensemble_clf, X_test, y_test, "Xception + Soft-Vote Ensemble")

print("\n" + "=" * 50)
print("ACCURACY COMPARISON")
print(f"  Xception + Random Forest  : {rf_acc:.4f}")
print(f"  Xception + Decision Tree  : {dt_acc:.4f}")
print(f"  Xception + SVM            : {svm_acc:.4f}")
print(f"  Xception + Ensemble       : {ens_acc:.4f}")
print("=" * 50)

## Section 8: Save Model

In [ ]:
joblib.dump(rf,  SAVED_MODELS_DIR + 'xception_rf.pkl')
joblib.dump(dt,  SAVED_MODELS_DIR + 'xception_dt.pkl')
joblib.dump(svm, SAVED_MODELS_DIR + 'xception_svm.pkl')
joblib.dump(ensemble_clf, SAVED_MODELS_DIR + 'xception_ensemble_model.pkl')
print(f"✓ Models saved to {SAVED_MODELS_DIR}")
print("  xception_rf.pkl, xception_dt.pkl, xception_svm.pkl, xception_ensemble_model.pkl")

## Section 9: Results Summary

In [ ]:
print("=" * 60)
print(f"NOTEBOOK: {NOTEBOOK_NAME}")
print(f"Dataset  : {train_generator.samples + val_generator.samples} training images")
print(f"Classes  : {CLASS_NAMES}")
print(f"Image size: {IMG_HEIGHT}×{IMG_WIDTH}")
print(f"Batch size: {BATCH_SIZE}, Seed: {RANDOM_SEED}")
print(f"Feature dim: {X_train.shape[1]} (Dense layer output)")
print("-" * 60)
print(f"  Xception + Random Forest  : {rf_acc:.4f}")
print(f"  Xception + Decision Tree  : {dt_acc:.4f}")
print(f"  Xception + SVM            : {svm_acc:.4f}")
print(f"  Xception + Ensemble       : {ens_acc:.4f}")
print("=" * 60)
print("Saved models location:", SAVED_MODELS_DIR)